We'll start with the import statements

In [1]:
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass


import os
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pytorch_lightning import Trainer


### NOTE: YOU WILL HAVE TO COMMENT THIS OUT FOR DDP ###
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using Device: {DEVICE}")

In [2]:
# JUST OUR USUAL DATASET CLASS
class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """Initializes the dataset object. Takes path and transform as inputs"""        
        super().__init__()        
        self.root_dir = root_dir
        self.transform = transform     
        self.classes = sorted(os.listdir(root_dir))        # Manually map folder names to integers
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        # Build the file list with classes
        self.images = []
        for cls_name in self.classes:
            cls_path = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_path):
                self.images.append((os.path.join(cls_path, img_name), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.images)         # Returns the number of images in the dataset

    def __getitem__(self, idx):
        """Grabs a sepcific image, loads it and applies transforms on it"""
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert("RGB")       
        if self.transform:
            image = self.transform(image)
            
        return image, label


In [3]:
### LOADING THE DATA USING DATALOADER
data_path = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset' ### Dataset folder path
## Adding transforms
mri_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Instantiate Datasets using the path of the data
train_dataset = BrainTumorDataset(os.path.join(data_path, 'Training'), transform=mri_transforms)
test_dataset = BrainTumorDataset(os.path.join(data_path, 'Testing'), transform=mri_transforms)

# Finally the DataLoader to create and load batches for model training. 
# Note how with a smaller dataset, loading a batch size of 32 is also feasible!
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count(), pin_memory=True)
val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)
print(f"Loaded {len(train_dataset)} training images across classes: {train_dataset.classes}")

Loaded 5600 training images across classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torchmetrics.classification import MulticlassAccuracy
import torchvision.models as models
import torch.nn as nn
import pytorch_lightning as pl

class BrainTumorClassifier(pl.LightningModule):
    """Define the model architecture, the forward pass, the train/val step and optimizer"""
    def __init__(self, num_classes=4):
        super().__init__()        
        backbone = models.resnet18(weights="DEFAULT") ## Resnet18 as backbone        
        num_filters = backbone.fc.in_features ### Extract the number of input features from the last layer (fc)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1]) # All except the last layer backbone
        ## Defining the final layer as classification head
        self.classifier = nn.Sequential(nn.Flatten(),nn.Linear(num_filters, 256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256, num_classes))
        self.train_acc = MulticlassAccuracy(num_classes=num_classes)
        self.val_acc = MulticlassAccuracy(num_classes=num_classes)

    def forward(self, x):
        x = self.feature_extractor(x) ## Using backbone as feature extractor and classification head on the output
        return self.classifier(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y) ## Calculate loss
        acc = self.train_acc(logits, y) ## Calculate accuracy   
        # For logs and display
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=True, on_epoch=True)      
        return loss    

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = self.val_acc(logits, y)       
        self.log("val_loss", loss, prog_bar=True)  # Adding prog_bar=True to see them in progress bar format
        self.log("val_acc", acc, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [5]:
model = BrainTumorClassifier(num_classes=4) ### Define the model


# Update the Trainer for Distributed Training here
# In devices specify the number of GPUs to use (2 or auto but don't use auto for this version)
# Set strategy to ddp_notebokk for Distributed Data Parallel in notebook. Else ddp
trainer = Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=2,  # Uses all available GPUs
    strategy="ddp_notebook",   # Enables PyTorch DDP
)

trainer.fit(model, train_loader, val_loader) ## Start distributed training process

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s] 
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
----------------------------------------------------------------------------------------------------
distributed_backen

┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ feature_extractor │ Sequential         │ 11.2 M │ train │     0 │
│ 1 │ classifier        │ Sequential         │  132 K │ train │     0 │
│ 2 │ train_acc         │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc           │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 11.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.3 M                                                                                               
Total estimated model params size (MB): 45                                                                         
Modules in train mode: 75                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream 
does not match the stream of the node that produced the incoming gradient. This may incur unnecessary 
synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This 
mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can 
happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which 
will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure 
that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, 
you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. 
(Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is 
recommended to use `self.log('train_loss', ..., sync_dist=True)` when logging on epoch level in distributed setting
to accumulate the metric across devices.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:433: It is 
recommended to use `self.log('train_acc', ..., sync_dist=True)` when logging on epoch level in distributed setting 
to accumulate the metric across devices.

`Trainer.fit` stopped: `max_epochs=10` reached.
